In [27]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
# Linear models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Support Vector Regression
from sklearn.svm import SVR

# Tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

# Neural Network
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

In [58]:
df = pd.read_csv('../data/processed/final_data_set.csv')

In [75]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,floor_cat
0,flat,sector 49,0.35,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,higher
1,flat,sector 49,0.38,2,2,2,not_available,1-5 Years Old,750.0,Basic,higher
2,flat,sector 92,0.49,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,lower
3,flat,sector 89,0.80,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,middle
4,flat,sector 49,1.55,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,middle


In [59]:
df.drop(columns=['Unnamed: 0' , 'bhk'] , inplace=True)

In [60]:
x = df.drop(columns=['price'])
y = df['price']

In [61]:
# Applying the log1p transformation to the target variable
y_log_transformed = np.log1p(y)

In [62]:
df.head().columns

Index(['property_type', 'sector', 'price', 'bedRoom', 'bathroom', 'balcony',
       'additionalRoom', 'agePossession', 'built_up_area', 'luxury_category',
       'floor_cat'],
      dtype='object')

In [63]:

# Creating a column Transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area']
        ),

        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            [
                'property_type',
                'balcony',
                'additionalRoom',
                'floor_cat',
                'luxury_category'
            ]
        ),

        (
            'cat1',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore'
                ,sparse_output=False
            ),
            ['sector', 'agePossession']
        )
    ],
    remainder='passthrough'
)

In [64]:

best_rf = RandomForestRegressor(
    n_estimators=600,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.7,
    max_depth=30,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', best_rf)
])

In [65]:
pipeline.fit(x,y_log_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
import pickle

with open('../models/model1.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [67]:
data = [[
    'flat',
    'sector 49',
    np.int64(3),
    np.int64(3),
    '3+',
    'servant room',
    'Upcoming',
    np.float64(1043.0),
    'Unfurnished',
    'higher'
]]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'additionalRoom', 'agePossession', 'built_up_area', 'luxury_category', 'floor_cat'] 

In [68]:
one_df = pd.DataFrame(data, columns=columns)
one_df

,property_type,sector,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,floor_cat
0,flat,sector 49,3,3,3+,servant room,Upcoming,1043.0,Unfurnished,higher


In [69]:
np.expm1(pipeline.predict(one_df))

array([1.23141396])

In [70]:
columns_cat= x.select_dtypes(include='object')

In [71]:
columns_cat.columns

Index(['property_type', 'sector', 'balcony', 'additionalRoom', 'agePossession',
       'luxury_category', 'floor_cat'],
      dtype='object')

In [72]:
columns_cat['floor_cat'].unique()

array(['higher', 'lower', 'middle'], dtype=object)

In [73]:
sectors = [
    'sector 49', 'sector 92', 'sector 89', 'sector 60', 'sector 95',
    'sector 82', 'sector 76', 'sector 37d', 'sector 102', 'sector 104',
    'sector 79', 'sector 37c', 'sector 22', 'sector 58', 'sector 90',
    'sector 69', 'sector 103', 'sector 28', 'sector 61', 'sector 50',
    'sector 36a', 'sector 81', 'sector 65', 'sector 86', 'sector 108',
    'sector 85', 'sector 2', 'sector 72', 'sector 70a', 'sector 111',
    'sector 107', 'sector 14', 'sector 99a', 'sector 109', 'sector 112',
    'sector 56', 'sector 21', 'sector 74', 'sector 31', 'sector 67a',
    'sector 82a', 'sector 66', 'sector 52', 'sector 110', 'sector 83',
    'sector 53', 'sector 70', 'sector 105', 'sector 33', 'sector 43',
    'sector 88a', 'sector 38', 'manesar', 'sector 99', 'sector 62',
    'sector 106', 'sector 1', 'sector 54', 'sector 67', 'sector 63a',
    'sector 3', 'sector 68', 'sector 23', 'sector 7', 'sector 113',
    'sector 12', 'sector 63', 'sector 84', 'dwarka expressway',
    'sector 59', 'sector 71', 'sector 4', 'sector 11', 'sector 55',
    'new sector 2', 'gwal pahari', 'sector 91', 'sector 93', 'sector 48',
    'sector 25', 'sector 47', 'new', 'sector 9a', 'sector 77',
    'sector 36', 'sector 30', 'sector 78', 'sector 80', 'sector 9',
    'sector 41', 'sector 6', 'sector 51', 'sector 39', 'sector 57',
    'sector 24', 'sector 17', 'sector 37', 'sector 13', 'sector 10a',
    'sector 8', 'sector 49 road', 'sector 45', 'sector 46', 'sector 40',
    'sector 5', 'sector 15', 'sector 26'
]

import re

def sector_number(x):
    match = re.search(r'\d+', x)
    return int(match.group()) if match else float('inf')

sorted_sectors = sorted(sectors, key=sector_number)

print(sorted_sectors)

['sector 1', 'sector 2', 'new sector 2', 'sector 3', 'sector 4', 'sector 5', 'sector 6', 'sector 7', 'sector 8', 'sector 9a', 'sector 9', 'sector 10a', 'sector 11', 'sector 12', 'sector 13', 'sector 14', 'sector 15', 'sector 17', 'sector 21', 'sector 22', 'sector 23', 'sector 24', 'sector 25', 'sector 26', 'sector 28', 'sector 30', 'sector 31', 'sector 33', 'sector 36a', 'sector 36', 'sector 37d', 'sector 37c', 'sector 37', 'sector 38', 'sector 39', 'sector 40', 'sector 41', 'sector 43', 'sector 45', 'sector 46', 'sector 47', 'sector 48', 'sector 49', 'sector 49 road', 'sector 50', 'sector 51', 'sector 52', 'sector 53', 'sector 54', 'sector 55', 'sector 56', 'sector 57', 'sector 58', 'sector 59', 'sector 60', 'sector 61', 'sector 62', 'sector 63a', 'sector 63', 'sector 65', 'sector 66', 'sector 67a', 'sector 67', 'sector 68', 'sector 69', 'sector 70a', 'sector 70', 'sector 71', 'sector 72', 'sector 74', 'sector 76', 'sector 77', 'sector 78', 'sector 79', 'sector 80', 'sector 81', 'sect

In [74]:
bedroom = df['bedRoom']
bhk = df['bhk']

KeyError: 'bhk'

np.float64(0.8365406699343743)